# 2 Preprocessing Data

Read in the Google Trends keyword interest data for preprocessing.

In [22]:
from pathlib import Path
import pandas as pd

In [36]:
DATA_DIR = Path.cwd()
TRENDS_PATH = DATA_DIR / "trends_final_keywords.csv"

trends = pd.read_csv(TRENDS_PATH, parse_dates=["date"])
trends.head()

,date,keyword,interest
0,2016-05-01,home care,100.0
1,2016-06-01,home care,60.0
2,2016-07-01,home care,90.0
3,2016-08-01,home care,100.0
4,2016-09-01,home care,90.0


In [40]:
# Pivot the table: keywords as rows, dates as columns, interest values as data
trends_pivot = trends.pivot_table(
    index="keyword", 
    columns="date", 
    values="interest", 
    aggfunc="first"
)

print(f"Shape after pivot: {trends_pivot.shape}")
print(f"Keywords (rows): {trends_pivot.shape[0]:,}")
print(f"Monthly observations (columns): {trends_pivot.shape[1]:,}")
print(f"Date range: {trends_pivot.columns.min().date()} to {trends_pivot.columns.max().date()}")

trends_pivot.head()

Shape after pivot: (1446, 121)
Keywords (rows): 1,446
Monthly observations (columns): 121
Date range: 2016-05-01 to 2026-05-01


date,2016-05-01,2016-06-01,2016-07-01,2016-08-01,2016-09-01,2016-10-01,2016-11-01,2016-12-01,2017-01-01,2017-02-01,...,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01,2026-04-01,2026-05-01
keyword,,,,,,,,,,,,,,,,,,,,,
5g,9.090909,6.666667,10.0,11.111111,10.0,10.0,4.0,8.333333,7.692308,7.692308,...,50.0,45.454545,34.482759,48.275862,56.0,62.962963,82.758621,41.025641,100.000,84.210526
5g network,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,3.448276,0.0,3.846154,3.333333,2.564103,3.125,0.000000
5g spectrum,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000,0.000000
5g stock,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000,0.000000
5g technology,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000,0.000000


In [41]:
# Filter 1: Remove keywords with 70% or more NaN values
nan_share_by_keyword = trends_pivot.isna().sum(axis=1) / len(trends_pivot.columns)
keywords_with_high_nan = nan_share_by_keyword[nan_share_by_keyword >= 0.70].index

print(f"Filter 1: NaN Removal (70%+ threshold)")
print(f"Keywords with 70%+ NaN: {len(keywords_with_high_nan):,}")

trends_pivot = trends_pivot.loc[~trends_pivot.index.isin(keywords_with_high_nan)]
print(f"Keywords remaining: {len(trends_pivot):,}")


Filter 1: NaN Removal (70%+ threshold)
Keywords with 70%+ NaN: 0
Keywords remaining: 1,446


In [42]:
# Filter 2: Remove keywords with 70% or more zero values
zero_share_by_keyword = (trends_pivot == 0).sum(axis=1) / len(trends_pivot.columns)
keywords_with_high_zeros = zero_share_by_keyword[zero_share_by_keyword >= 0.70].index

print(f"\nFilter 2: Zero Removal (70%+ threshold)")
print(f"Keywords with 70%+ zeros: {len(keywords_with_high_zeros):,}")

trends_pivot = trends_pivot.loc[~trends_pivot.index.isin(keywords_with_high_zeros)]
print(f"Keywords remaining: {len(trends_pivot):,}")



Filter 2: Zero Removal (70%+ threshold)
Keywords with 70%+ zeros: 967
Keywords remaining: 479


In [43]:
# Check mean interest statistics AFTER filtering
mean_interest_by_keyword = trends_pivot.mean(axis=1)

print(f"\nMean Interest Statistics (after NaN and zero filtering):")
print(f"Min: {mean_interest_by_keyword.min():.2f}")
print(f"Max: {mean_interest_by_keyword.max():.2f}")
print(f"Mean: {mean_interest_by_keyword.mean():.2f}")
print(f"Median: {mean_interest_by_keyword.median():.2f}")

# Summary table for different thresholds
thresholds = [0.5, 1, 2, 5, 10, 20, 50]
mean_filter_summary = []

for threshold in thresholds:
    keywords_below_threshold = mean_interest_by_keyword[mean_interest_by_keyword < threshold]
    keywords_remaining = len(trends_pivot) - len(keywords_below_threshold)
    
    mean_filter_summary.append({
        "mean_interest_threshold": threshold,
        "keywords_dropped": len(keywords_below_threshold),
        "keywords_remaining": keywords_remaining,
    })

mean_filter_summary_df = pd.DataFrame(mean_filter_summary)
print(f"\nMean Interest Filtering Summary:")
print(mean_filter_summary_df.to_string(index=False))



Mean Interest Statistics (after NaN and zero filtering):
Min: 1.27
Max: 3742.46
Mean: 64.20
Median: 10.29

Mean Interest Filtering Summary:
 mean_interest_threshold  keywords_dropped  keywords_remaining
                     0.5                 0                 479
                     1.0                 0                 479
                     2.0                14                 465
                     5.0                91                 388
                    10.0               234                 245
                    20.0               319                 160
                    50.0               394                  85


In [44]:
# Filter 3 (Optional): Apply mean interest threshold
mean_interest_threshold = 10

keywords_below_threshold = mean_interest_by_keyword[mean_interest_by_keyword < mean_interest_threshold].index

print(f"\nFilter 3: Mean Interest Threshold ({mean_interest_threshold})")
print(f"Keywords with mean interest < {mean_interest_threshold}: {len(keywords_below_threshold):,}")

trends_pivot = trends_pivot.loc[~trends_pivot.index.isin(keywords_below_threshold)]

print(f"Keywords remaining: {len(trends_pivot):,}")
print(f"\nFinal shape: {trends_pivot.shape}")
print(f"Keywords (rows): {trends_pivot.shape[0]:,}")
print(f"Time periods (columns): {trends_pivot.shape[1]:,}")



Filter 3: Mean Interest Threshold (10)
Keywords with mean interest < 10: 234
Keywords remaining: 245

Final shape: (245, 121)
Keywords (rows): 245
Time periods (columns): 121


In [45]:
# Display all keywords for manual review
# Company names should be removed, but "company stock" is OK
print("=" * 80)
print("KEYWORDS FOR REVIEW (245 total)")
print("=" * 80)
print("\nRemove standalone company names, but keep '[company] stock' terms\n")

all_keywords = trends_pivot.index.tolist()

for i, keyword in enumerate(all_keywords, 1):
    print(f"{i:3d}. {keyword}")

# Save to CSV for easier review in a spreadsheet
keywords_for_review = pd.DataFrame({
    'keyword': all_keywords,
    'keep': [''] * len(all_keywords),  # Column for manual review
})
keywords_for_review.to_csv(DATA_DIR / 'keywords_for_review.csv', index=False)
print(f"\n\nKeywords saved to 'keywords_for_review.csv' for manual review")


KEYWORDS FOR REVIEW (245 total)

Remove standalone company names, but keep '[company] stock' terms

  1. 5g
  2. advertising
  3. aerospace
  4. airlines
  5. alphabet
  6. aluminum
  7. amazon stock
  8. amd stock
  9. api
 10. apple iphone
 11. apple stock
 12. asset management
 13. auto parts
 14. auto sales
 15. automation
 16. balance sheet
 17. bank capital
 18. bank of america
 19. banking
 20. banks
 21. benefits
 22. berkshire hathaway
 23. black friday
 24. broadband
 25. brokerage
 26. cable tv
 27. cancer treatment
 28. capital gains
 29. carbon fiber
 30. casinos
 31. cement
 32. central bank
 33. charity
 34. chemicals
 35. chevron
 36. clay
 37. climate change
 38. coal
 39. comcast
 40. commercial property
 41. commercial real estate
 42. computing
 43. construction
 44. copper
 45. cost of living
 46. cpi
 47. credit card
 48. credit cards
 49. credit score
 50. cryptocurrency
 51. currency
 52. cyber monday
 53. cyber security
 54. data center
 55. defense
 56. depres

In [46]:
# Remove standalone company names (keep "company stock" terms)
company_names_to_remove = [
    'alphabet',
    'berkshire hathaway',
    'chevron',
    'comcast',
    'disney plus',
    'disney+',
    'exxon',
    'goldman sachs',
    'home depot',
    'lockheed martin',
    'meta',
    'netflix',
    'paramount plus',
    'redfin',
    'spotify',
    'tiktok',
    'walmart grocery',
    'wells fargo',
    'zillow',
]

print("Removing standalone company names:")
print("=" * 60)

keywords_before = len(trends_pivot)
trends_pivot = trends_pivot.loc[~trends_pivot.index.isin(company_names_to_remove)]
keywords_removed = keywords_before - len(trends_pivot)

for company in company_names_to_remove:
    if company in all_keywords:
        print(f"  ✓ Removed: {company}")

print("=" * 60)
print(f"Companies removed: {keywords_removed}")
print(f"Keywords remaining: {len(trends_pivot):,}")
print(f"\nFinal shape: {trends_pivot.shape}")


Removing standalone company names:
  ✓ Removed: alphabet
  ✓ Removed: berkshire hathaway
  ✓ Removed: chevron
  ✓ Removed: comcast
  ✓ Removed: disney plus
  ✓ Removed: disney+
  ✓ Removed: exxon
  ✓ Removed: goldman sachs
  ✓ Removed: home depot
  ✓ Removed: lockheed martin
  ✓ Removed: meta
  ✓ Removed: netflix
  ✓ Removed: paramount plus
  ✓ Removed: redfin
  ✓ Removed: spotify
  ✓ Removed: tiktok
  ✓ Removed: walmart grocery
  ✓ Removed: wells fargo
  ✓ Removed: zillow
Companies removed: 19
Keywords remaining: 226

Final shape: (226, 121)


In [47]:
# Save the final filtered dataframe as CSV
output_path = DATA_DIR / 'trends_final_filtered.csv'
trends_pivot.to_csv(output_path)

print(f"Final filtered dataset saved to: trends_final_filtered.csv")
print(f"\nDataset summary:")
print(f"  Shape: {trends_pivot.shape}")
print(f"  Keywords: {trends_pivot.shape[0]:,}")
print(f"  Time periods: {trends_pivot.shape[1]:,}")
print(f"  Date range: {trends_pivot.columns.min().date()} to {trends_pivot.columns.max().date()}")
print(f"\nReady for PCA analysis!")


Final filtered dataset saved to: trends_final_filtered.csv

Dataset summary:
  Shape: (226, 121)
  Keywords: 226
  Time periods: 121
  Date range: 2016-05-01 to 2026-05-01

Ready for PCA analysis!


In [ ]:
# Check keywords with highest average interest
mean_interest = trends_pivot.mean(axis=1).sort_values(ascending=False)

print("Top 20 keywords by mean interest:")
print(mean_interest.head(20))

print("\n\nBottom 20 keywords by mean interest:")
print(mean_interest.tail(20))

print(f"\n\nMean interest statistics:")
print(f"Max: {mean_interest.max():.2f}")
print(f"Min: {mean_interest.min():.2f}")
print(f"Mean: {mean_interest.mean():.2f}")
print(f"Median: {mean_interest.median():.2f}")
print(f"Std: {mean_interest.std():.2f}")
